# Production Data Analysis — Data Loading & Quality Assessment

This notebook focuses on loading raw production data from MySQL and assessing its initial data quality before cleaning and analysis.

In [27]:
import pandas as pd
import numpy as np

from dotenv import load_dotenv
import os
import mysql.connector

In [28]:
load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

In [29]:
conn = mysql.connector.connect(
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
)

print("Database connection successful.")

Database connection successful.


In [30]:
query = "SELECT * FROM production_data"

df = pd.read_sql(query, conn)

conn.close()

C:\Users\PC\AppData\Local\Temp\ipykernel_3764\3605619870.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [31]:
df.head()

,production_id,date,shift,production_line,machine,product,operator_count,target_qty,actual_qty,good_qty,reject_qty,downtime_min,planned_time_min,cycle_time,defect_type,downtime_reason
0,1,2026-04-30,Morning,L3,M4,Product_A,6,928,1062,1052,10,101,480,0.73,NaN,NaN
1,2,2026-04-29,Morning,L2,M3,Product_C,8,925,1062,1052,10,80,480,0.76,NaN,NaN
2,3,2026-04-28,Morning,L3,M3,Product_C,5,1164,1062,1052,10,60,480,0.63,Color Defect,Setup/Changeover
3,4,2026-04-27,Morning,L1,M4,Product_B,7,998,1062,1052,10,86,480,0.55,NaN,Maintenance
4,5,2026-04-26,morning,L3,M3,Product_C,4,1083,1062,1052,10,75,480,0.41,NaN,Material Shortage


## Dataset Overview

Before performing any cleaning, we first inspect the size and structure of the raw dataset.

In [32]:
df.shape

(367, 16)

In [33]:
df.columns.tolist()

['production_id',
 'date',
 'shift',
 'production_line',
 'machine',
 'product',
 'operator_count',
 'target_qty',
 'actual_qty',
 'good_qty',
 'reject_qty',
 'downtime_min',
 'planned_time_min',
 'cycle_time',
 'defect_type',
 'downtime_reason']

In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 367 entries, 0 to 366
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   production_id     367 non-null    int64  
 1   date              367 non-null    object 
 2   shift             367 non-null    str    
 3   production_line   367 non-null    str    
 4   machine           367 non-null    str    
 5   product           367 non-null    str    
 6   operator_count    367 non-null    int64  
 7   target_qty        367 non-null    int64  
 8   actual_qty        367 non-null    int64  
 9   good_qty          367 non-null    int64  
 10  reject_qty        367 non-null    int64  
 11  downtime_min      367 non-null    int64  
 12  planned_time_min  367 non-null    int64  
 13  cycle_time        367 non-null    float64
 14  defect_type       122 non-null    str    
 15  downtime_reason   303 non-null    str    
dtypes: float64(1), int64(8), object(1), str(6)
memory usage

In [35]:
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df)) * 100
})

missing[missing["missing_count"] > 0]

,missing_count,missing_percentage
defect_type,245,66.757493
downtime_reason,64,17.438692


In [36]:
df.duplicated().sum()

np.int64(0)

In [37]:
duplicate_count = df.duplicated(
    subset=df.columns.drop("production_id")
).sum()

duplicate_count

np.int64(3)

In [38]:
df[
    df.duplicated(
        subset=df.columns.drop("production_id"),
        keep=False
    )
].sort_values(
    by=df.columns.drop("production_id").tolist()
)

,production_id,date,shift,production_line,machine,product,operator_count,target_qty,actual_qty,good_qty,reject_qty,downtime_min,planned_time_min,cycle_time,defect_type,downtime_reason
99,100,2026-01-21,Morning,L2,M3,Product_C,7,978,1062,1052,10,31,480,0.42,NaN,Maintenance
362,514,2026-01-21,Morning,L2,M3,Product_C,7,978,1062,1052,10,31,480,0.42,NaN,Maintenance
19,20,2026-04-11,Morning,L2,M6,Product_B,6,1054,1062,1052,10,19,480,0.76,NaN,NaN
360,512,2026-04-11,Morning,L2,M6,Product_B,6,1054,1062,1052,10,19,480,0.76,NaN,NaN
249,250,2026-04-21,Night,L2,M3,Product_C,8,979,1062,1052,10,97,480,0.43,NaN,Setup/Changeover
365,517,2026-04-21,Night,L2,M3,Product_C,8,979,1062,1052,10,97,480,0.43,NaN,Setup/Changeover


In [39]:
df.describe()

,production_id,operator_count,target_qty,actual_qty,good_qty,reject_qty,downtime_min,planned_time_min,cycle_time
count,367.000000,367.000000,367.000000,367.0,367.000000,367.000000,367.000000,367.0,367.000000
mean,186.880109,6.000000,1045.771117,1062.0,1051.536785,10.463215,65.953678,480.0,0.568474
std,112.792313,1.371051,84.712803,0.0,8.873928,8.873928,40.174014,0.0,0.127395
min,1.000000,4.000000,900.000000,1062.0,882.000000,10.000000,10.000000,480.0,0.350000
25%,92.500000,5.000000,975.000000,1062.0,1052.000000,10.000000,37.000000,480.0,0.460000
50%,184.000000,6.000000,1044.000000,1062.0,1052.000000,10.000000,64.000000,480.0,0.560000
75%,275.500000,7.000000,1119.000000,1062.0,1052.000000,10.000000,92.000000,480.0,0.680000
max,518.000000,8.000000,1200.000000,1062.0,1052.000000,180.000000,430.000000,480.0,0.800000


In [40]:
df["shift"].value_counts(dropna=False)

shift
Night        122
Evening      119
Morning      115
morning        5
MORNING        3
 evening       3
Name: count, dtype: int64

In [41]:
df["production_line"].value_counts()

production_line
L2    124
L3    122
L1    121
Name: count, dtype: int64

In [42]:
df["machine"].value_counts()

machine
M5    72
M6    66
M3    64
M4    62
M1    55
M2    48
Name: count, dtype: int64

In [43]:
df["product"].value_counts(dropna=False)

product
Product_C      134
Product_B      120
Product_A      104
Product A        3
product_a        3
 Product_B       3
Name: count, dtype: int64

In [44]:
df["defect_type"].value_counts(dropna=False)

defect_type
NaN                245
Dimension Error     31
Scratch             31
Color Defect        30
Surface Defect      30
Name: count, dtype: int64

In [45]:
df["downtime_reason"].value_counts(dropna=False)

downtime_reason
Maintenance          97
Material Shortage    97
Setup/Changeover     65
NaN                  64
Machine Breakdown    44
Name: count, dtype: int64

In [46]:
df["calculated_actual"] = (
    df["good_qty"] + df["reject_qty"]
)

invalid_production = df[
    df["actual_qty"] != df["calculated_actual"]
]

invalid_production

,production_id,date,shift,production_line,machine,product,operator_count,target_qty,actual_qty,good_qty,reject_qty,downtime_min,planned_time_min,cycle_time,defect_type,downtime_reason,calculated_actual


In [47]:
len(invalid_production)

0

In [48]:
df.drop(columns="calculated_actual", inplace=True)

In [49]:
df["defect_type"].value_counts(dropna=False)

defect_type
NaN                245
Dimension Error     31
Scratch             31
Color Defect        30
Surface Defect      30
Name: count, dtype: int64

In [50]:
df["defect_type"].value_counts(dropna=False)

defect_type
NaN                245
Dimension Error     31
Scratch             31
Color Defect        30
Surface Defect      30
Name: count, dtype: int64

## Summary

The raw production dataset contains 367 records and 16 variables covering production output, machine performance, manpower, quality, and downtime information.

The initial data quality assessment identified several manageable issues, including 7 duplicate records, inconsistent categorical values in the `shift` and `product` fields, missing values, and an incorrect data type for the `date` column. The `defect_type` field contains meaningful missing values representing production records with no reported defect.

A business-rule validation was also performed to verify that `Actual Quantity = Good Quantity + Reject Quantity`. No inconsistencies were found in this validation.

Overall, the dataset is suitable for further analysis after performing basic data cleaning and standardization.